# Re-test a finished CoTrainingEnsemble_v2 run

Rebuilds a trained ensemble from its saved checkpoints and re-scores it on the test split,
with no retraining.

`load_ensemble_for_inference` reads the architecture spec embedded in each checkpoint, so the
models do **not** have to be re-declared by hand. For checkpoints written before that spec
existed, pass the run's `<model_version>-scania.csv` as `results_csv_path` so the ensemble
weights can be recovered from its `weight_h*` columns (see the last section).

In [1]:
import json
import os

import torch
from torchmetrics.functional import mean_squared_error

from scania.dataset import ScaniaDataModule
from scania.metrics import scania_score
from scania.utils import load_ensemble_for_inference
from shared.utils import ModelVersion, set_seed

In [2]:
RUN = "model-co_training_ensemble_v2-scania-2026-08-10_16-39-23"

CHECKPOINTS_PATH = os.path.join("checkpoints", RUN)
RESULTS_PATH = os.path.join("outputs", "co_training_ensemble_v2_with_scania_score", RUN)
SCORES_CSV = os.path.join(RESULTS_PATH, "co_training_ensemble_v2-scania.csv")

# The run was trained on Colab, so run_parameters.json points at Google Drive paths.
# Override just those with the local ones; everything else must stay identical or the test
# split will not be the same one the run was scored on.
LOCAL_DATA_DIR = os.path.join("data", "Scania_component_X")
# A dedicated cache dir: ScaniaDataModule keeps one cache per directory and rebuilds it in
# place whenever the config differs, and the existing data/Scania_component_X/scania_cache was
# built with data_fraction=0.2 while this run used 1.0 -- reusing it would overwrite it.
# The first run of this notebook therefore rebuilds the splits from the raw CSVs (slow once,
# cached afterwards).
LOCAL_CACHE_DIR = os.path.join(LOCAL_DATA_DIR, "scania_cache_eval")

run_parameters = json.load(open(os.path.join(RESULTS_PATH, "run_parameters.json"), encoding="utf-8"))
dataset_parameters = dict(run_parameters["dataset_parameters"])
dataset_parameters.update({
    "data_dir": LOCAL_DATA_DIR,
    "cache_dir": LOCAL_CACHE_DIR,
    "num_workers": 0,
    "pin_memory": False,
})
dataset_parameters

{'data_dir': 'data\\Scania_component_X',
 'seed': 42,
 'data_fraction': 1.0,
 'val_rate': 0.15,
 'test_rate': 0.15,
 'calib_rate': 0.1,
 'stratify': True,
 'norm_type': 'z-score',
 'shuffle_loader': True,
 'cache_dir': 'data\\Scania_component_X\\scania_cache_eval',
 'num_workers': 0,
 'pin_memory': False,
 'return_sequence_label': False,
 'batch_size': 128,
 'sequence_len': 48,
 'counter_mode': 'both',
 'include_histograms': False,
 'histogram_mode': 'zhist',
 'force_load_from_cache': False}

## Test split

The same seed and split rates as the run, so `get_cotraining_tensors("test")` returns the
exact rows the saved predictions were computed on.

In [3]:
set_seed(dataset_parameters["seed"])

data_module = ScaniaDataModule(**dataset_parameters)
data_module.setup()

test_features, test_targets, _, _ = data_module.get_cotraining_tensors("test")
print(f"feature_num = {len(data_module.feature_cols)}")
print(f"test features {tuple(test_features.shape)} | targets {tuple(test_targets.shape)}")

[Scania] No valid cache found, preprocessing from raw files...
[Scania] Preprocessing done in 12.8s | vehicles train/val/test/calib = 14144/3528/3528/2350
[Scania] Cache written to data\Scania_component_X\scania_cache_eval
feature_num = 16
test features (339, 48, 16) | targets (339, 1)


## Load the ensemble

In [4]:
ensemble = load_ensemble_for_inference(
    checkpoints_path=CHECKPOINTS_PATH,
    model_version=ModelVersion.CO_TRAINING_ENSEMBLE_V2,
    # Only needed for legacy checkpoints; harmless (and ignored) once the checkpoints carry
    # their own weights.
    results_csv_path=SCORES_CSV,
    map_location="cuda" if torch.cuda.is_available() else "cpu",
    inference_batch_size=1024,
)

for i, module in enumerate(ensemble.lightning_modules):
    print(f"h{i}: {type(module.net).__name__:<26} weight={ensemble.weights[i]:.4f} "
          f"target_mean={module.target_mean:.3f} target_std={module.target_std:.3f}")

h0: CNN1D                      weight=0.2467 target_mean=62.187 target_std=62.018
h1: Simple_LSTM                weight=0.2431 target_mean=62.187 target_std=62.018
h2: TransformerFeatures        weight=0.2489 target_mean=62.187 target_std=62.018
h3: TransformerTimeSequence    weight=0.2614 target_mean=62.187 target_std=62.018


## Re-score

`predict` / `predict_per_model` go through `BasicLightningModule.forward`, which de-normalizes,
so these predictions are already in real RUL units — directly comparable to `test_targets`.

In [5]:
targets_flat = test_targets.detach().cpu().view(-1)


def evaluate(predictions: torch.Tensor) -> tuple[float, float]:
    """Return (RMSE, Scania cost) for one prediction tensor against the test targets."""
    predictions = predictions.detach().cpu().view(-1)
    rmse = float(mean_squared_error(predictions, targets_flat, squared=False))
    return rmse, float(scania_score(predictions.numpy(), targets_flat.numpy()))


with torch.no_grad():
    per_model = ensemble.predict_per_model(test_features)
    weighted = ensemble.predict(test_features)

for i, predictions in enumerate(per_model):
    rmse, score = evaluate(predictions)
    print(f"h{i:<2} RMSE {rmse:8.4f} | score {score:9.1f}")

rmse_weighted, score_weighted = evaluate(weighted)
print(f"{'weighted':<4} RMSE {rmse_weighted:8.4f} | score {score_weighted:9.1f}")

h0  RMSE  34.4615 | score   56860.0
h1  RMSE  37.1145 | score   58479.0
h2  RMSE  34.1289 | score   83100.0
h3  RMSE  37.6289 | score   69210.0
weighted RMSE  32.8506 | score   68088.0


## Compare against the run's saved scores

A match confirms the reload is faithful: same weights, same architectures, same target
standardization stats.

In [6]:
import pandas as pd

saved = pd.read_csv(SCORES_CSV).loc[0]
rows = []
for i, predictions in enumerate(per_model):
    rmse, score = evaluate(predictions)
    rows.append({"model": f"h{i}", "rmse": rmse, "saved_rmse": saved[f"test_rmse_h{i}"],
                 "score": score, "saved_score": saved[f"test_score_h{i}"]})
rows.append({"model": "weighted", "rmse": rmse_weighted, "saved_rmse": saved["test_rmse_weighted"],
             "score": score_weighted, "saved_score": saved["test_score_weighted"]})

comparison = pd.DataFrame(rows)
comparison["rmse_delta"] = comparison["rmse"] - comparison["saved_rmse"]
comparison

,model,rmse,saved_rmse,score,saved_score,rmse_delta
0,h0,34.461494,34.461494,56860.0,56860.0,0.000000
1,h1,37.114452,37.114456,58479.0,58479.0,-0.000004
2,h2,34.128922,34.128922,83100.0,83100.0,0.000000
3,h3,37.628872,37.628868,69210.0,69210.0,0.000004
4,weighted,32.850636,32.850636,68088.0,68088.0,0.000000


## Loading a single model by hand

`save_ensemble_outputs` now writes real Lightning checkpoints, so
`BasicLightningModule.load_from_checkpoint` works directly. It still needs the architecture
(`model=`), because `save_hyperparameters(ignore=['model'])` deliberately keeps the `nn.Module`
out of the hyperparameters — `load_module_checkpoint` is the shortcut that rebuilds it from the
checkpoint's embedded spec.

In [7]:
from scania.lightning_module import BasicLightningModule
from scania.utils import load_module_checkpoint, read_checkpoint_spec
from models import CNN1D

cnn_ckpt = os.path.join(CHECKPOINTS_PATH, "co_training_ensemble_v2_cnn_0.pth")
print("spec:", read_checkpoint_spec(cnn_ckpt))

# Rebuilt from the embedded spec.
module = load_module_checkpoint(cnn_ckpt)

# Equivalent, with the architecture supplied explicitly (note: feature_num comes from the data
# module, it is not 8 -- counter_mode='both' doubles the counter columns).
module_explicit = BasicLightningModule.load_from_checkpoint(
    checkpoint_path=cnn_ckpt,
    model=CNN1D(num_features=len(data_module.feature_cols)),
    map_location="cpu",
)

with torch.no_grad():
    sample = test_features[:8].cpu()
    print(torch.allclose(module.cpu()(sample), module_explicit.eval()(sample), atol=1e-6))

spec: None


C:\Users\Epulapp\PycharmProjects\SurvivalAnalysisScaniaComponentX\.venv\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


TypeError: 'BasicLightningModule' object is not subscriptable